# Week 3: Entropy-Driven Interactive Generation

## The User's Vision

During generation:
1. **Generate tokens sequentially**
2. **At each step, compute entropy H**
3. **If H > threshold** (model is uncertain):
   - Extract hidden state
   - Probe classifies: **code or language uncertainty?**
   - If **CODE**: Stop and ask user for clarification ❗
   - If **LANGUAGE**: Continue (just word choice, not critical) ✓
4. **If H ≤ threshold**: Model is confident, continue ✓

## Why This Works

- **Code uncertainty**: Model doesn't know which API/function/module to use → needs user input
- **Language uncertainty**: Model doesn't know which word/phrase to use → can pick any reasonable option
- **Interactive stopping**: Only interrupt when it matters (missing code context)

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
import pickle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

In [ ]:
# Cell 4: Load trained classifier

print("Loading uncertainty type classifier...")
with open('uncertainty_type_classifier.pkl', 'rb') as f:
    model_data = pickle.load(f)

probe = model_data['probe']
scaler = model_data['scaler']
SELECTED_LAYERS = model_data['layers']

print(f"✅ Classifier loaded")
print(f"   Layers: {SELECTED_LAYERS}")
print(f"   Training accuracy: {model_data['train_accuracy']:.1%}")
print(f"   CV accuracy: {model_data['cv_accuracy']:.1%}")
print(f"   Training examples: {model_data['n_train_examples']}")

## 1. Helper Functions

In [ ]:
# Cell 5: Helper functions

def get_multi_layer_state(prompt: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    """Compute entropy in bits."""
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def classify_uncertainty_type(prompt: str) -> Tuple[int, float]:
    """
    Classify uncertainty type for the given prompt.
    
    Returns:
        (uncertainty_type, probability)
        uncertainty_type: 1 = code, 0 = language
        probability: P(code uncertainty)
    """
    h = get_multi_layer_state(prompt, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    
    uncertainty_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    
    return int(uncertainty_type), float(probability)

print("✅ Helper functions ready")

## 2. Entropy-Driven Generation

In [ ]:
# Cell 6: Main generation function

def generate_with_entropy_monitoring(
    prompt: str,
    entropy_threshold: float = 3.0,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Generate text with entropy monitoring and uncertainty classification.
    
    Args:
        prompt: Input prompt
        entropy_threshold: H > threshold triggers uncertainty check
        max_tokens: Maximum tokens to generate
        verbose: Print detailed logs
    
    Returns:
        Dictionary with generation results and stopping info
    """
    current_text = prompt
    generated_tokens = []
    entropy_trace = []
    stop_reason = None
    stop_info = {}
    
    if verbose:
        print(f"\nStarting generation with entropy threshold: {entropy_threshold:.1f} bits")
        print(f"Prompt: '{prompt}'")
        print("="*80)
    
    for step in range(max_tokens):
        # Get next token distribution
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()
        
        probs = softmax(logits)
        
        # Compute entropy
        H = entropy_from_probs(probs)
        entropy_trace.append(H)
        
        # Sample next token (greedy for reproducibility)
        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])
        
        if verbose:
            print(f"\nStep {step + 1}:")
            print(f"  Next token: '{next_token}'")
            print(f"  Entropy: {H:.3f} bits")
        
        # Check if entropy exceeds threshold (model is uncertain)
        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY ({H:.3f} > {entropy_threshold:.1f})")
                print(f"  Classifying uncertainty type...")
            
            # Classify uncertainty type
            uncertainty_type, prob = classify_uncertainty_type(current_text)
            
            if verbose:
                type_str = "CODE" if uncertainty_type == 1 else "LANGUAGE"
                print(f"  Uncertainty type: {type_str} (P={prob:.3f})")
            
            if uncertainty_type == 1:  # Code uncertainty
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY DETECTED")
                    print(f"  Stopping generation - need user clarification!")
                
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'uncertainty_type': 'code',
                    'uncertainty_prob': prob,
                    'next_token_would_be': next_token
                }
                break
            else:  # Language uncertainty
                if verbose:
                    print(f"  ✓ Language uncertainty - continuing (just word choice)")
        else:
            if verbose:
                print(f"  ✓ Low entropy - model is confident")
        
        # Add token to generated text
        generated_tokens.append(next_token)
        current_text += next_token
        
        # Check for EOS
        if next_token_id == tokenizer.eos_token_id:
            if verbose:
                print(f"  EOS token reached")
            stop_reason = "eos"
            break
    
    if stop_reason is None:
        stop_reason = "max_tokens"
        if verbose:
            print(f"\nReached max tokens ({max_tokens})")
    
    generated_text = ''.join(generated_tokens)
    
    if verbose:
        print(f"\n" + "="*80)
        print(f"GENERATION COMPLETE")
        print(f"Stop reason: {stop_reason}")
        print(f"Tokens generated: {len(generated_tokens)}")
        print(f"\nFinal text: '{current_text}'")
    
    return {
        'prompt': prompt,
        'generated_text': generated_text,
        'full_text': current_text,
        'generated_tokens': generated_tokens,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_tokens)
    }

print("✅ Generation function ready")

## 3. Test Examples

In [ ]:
# Cell 7: Test on code uncertainty examples

print("="*80)
print("TEST 1: CODE UNCERTAINTY (Should stop and ask for clarification)")
print("="*80)

code_examples = [
    "import",
    "from sklearn import",
    "def calculate_",
]

code_results = []
for prompt in code_examples:
    result = generate_with_entropy_monitoring(
        prompt,
        entropy_threshold=3.0,
        max_tokens=20,
        verbose=True
    )
    code_results.append(result)
    print("\n" + "="*80 + "\n")

In [ ]:
# Cell 8: Test on language uncertainty examples

print("="*80)
print("TEST 2: LANGUAGE UNCERTAINTY (Should continue generation)")
print("="*80)

lang_examples = [
    "The algorithm is",
    "Write a poem about",
    "In conclusion,",
]

lang_results = []
for prompt in lang_examples:
    result = generate_with_entropy_monitoring(
        prompt,
        entropy_threshold=3.0,
        max_tokens=20,
        verbose=True
    )
    lang_results.append(result)
    print("\n" + "="*80 + "\n")

In [ ]:
# Cell 9: Test on low entropy examples

print("="*80)
print("TEST 3: LOW ENTROPY (Model is confident, should continue)")
print("="*80)

confident_examples = [
    "The sky is",
    "2 + 2 =",
]

confident_results = []
for prompt in confident_examples:
    result = generate_with_entropy_monitoring(
        prompt,
        entropy_threshold=3.0,
        max_tokens=20,
        verbose=True
    )
    confident_results.append(result)
    print("\n" + "="*80 + "\n")

## 4. Analysis

In [ ]:
# Cell 10: Summary analysis

all_results = code_results + lang_results + confident_results

print("\n" + "="*80)
print("SUMMARY ANALYSIS")
print("="*80)

print(f"\nTotal examples tested: {len(all_results)}")

# Count stop reasons
stop_reasons = {}
for result in all_results:
    reason = result['stop_reason']
    stop_reasons[reason] = stop_reasons.get(reason, 0) + 1

print(f"\nStop reasons:")
for reason, count in stop_reasons.items():
    print(f"  {reason}: {count}")

# Code uncertainty detection
code_stopped = sum(1 for r in code_results if r['stop_reason'] == 'code_uncertainty')
print(f"\nCode uncertainty examples:")
print(f"  Stopped for code uncertainty: {code_stopped}/{len(code_results)} ({code_stopped/len(code_results):.0%})")

# Language uncertainty - should NOT stop
lang_stopped = sum(1 for r in lang_results if r['stop_reason'] == 'code_uncertainty')
print(f"\nLanguage uncertainty examples:")
print(f"  Incorrectly stopped: {lang_stopped}/{len(lang_results)}")

# Average entropy
avg_entropy_code = np.mean([np.mean(r['entropy_trace']) for r in code_results])
avg_entropy_lang = np.mean([np.mean(r['entropy_trace']) for r in lang_results])
avg_entropy_conf = np.mean([np.mean(r['entropy_trace']) for r in confident_results])

print(f"\nAverage entropy:")
print(f"  Code uncertainty: {avg_entropy_code:.3f} bits")
print(f"  Language uncertainty: {avg_entropy_lang:.3f} bits")
print(f"  Confident: {avg_entropy_conf:.3f} bits")

In [ ]:
# Cell 11: Detailed results table

print("\n" + "="*100)
print("DETAILED RESULTS")
print("="*100)

print(f"\n{'Prompt':<30} {'Stop Reason':<20} {'Steps':<8} {'Avg H':<10} {'Stop Info'}")
print("-"*100)

for result in all_results:
    prompt_short = result['prompt'][:30]
    avg_H = np.mean(result['entropy_trace'])
    
    stop_info_str = ""
    if result['stop_reason'] == 'code_uncertainty':
        info = result['stop_info']
        stop_info_str = f"H={info['entropy']:.2f}, P(code)={info['uncertainty_prob']:.2f}"
    
    print(f"{prompt_short:<30} {result['stop_reason']:<20} {result['num_steps']:<8} {avg_H:<10.3f} {stop_info_str}")

In [ ]:
# Cell 12: Visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Entropy traces for code examples
ax = axes[0, 0]
for i, result in enumerate(code_results):
    ax.plot(result['entropy_trace'], label=result['prompt'][:20], marker='o')
ax.axhline(3.0, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.set_xlabel('Generation Step')
ax.set_ylabel('Entropy (bits)')
ax.set_title('Entropy Trace: Code Uncertainty Examples')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Entropy traces for language examples
ax = axes[0, 1]
for i, result in enumerate(lang_results):
    ax.plot(result['entropy_trace'], label=result['prompt'][:20], marker='o')
ax.axhline(3.0, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.set_xlabel('Generation Step')
ax.set_ylabel('Entropy (bits)')
ax.set_title('Entropy Trace: Language Uncertainty Examples')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Average entropy comparison
ax = axes[1, 0]
categories = ['Code\nUncertainty', 'Language\nUncertainty', 'Confident']
avg_entropies = [avg_entropy_code, avg_entropy_lang, avg_entropy_conf]
colors_bar = ['coral', 'steelblue', 'green']
bars = ax.bar(categories, avg_entropies, color=colors_bar, alpha=0.7)
ax.axhline(3.0, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.set_ylabel('Average Entropy (bits)')
ax.set_title('Average Entropy by Category')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, avg_entropies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontsize=10)

# Plot 4: Stop reasons distribution
ax = axes[1, 1]
reasons = list(stop_reasons.keys())
counts = [stop_reasons[r] for r in reasons]
colors_pie = ['red', 'orange', 'green']
ax.pie(counts, labels=reasons, autopct='%1.0f%%', startangle=90, colors=colors_pie[:len(reasons)])
ax.set_title('Distribution of Stop Reasons')

plt.tight_layout()
plt.savefig('interactive_generation_results.png', dpi=150)
plt.show()

print("\n✅ Visualizations saved")

## 5. Interactive Demo

In [ ]:
# Cell 13: Interactive demo

def interactive_generation_demo():
    """
    Interactive demo where user can input prompts and see
    the entropy-driven generation in action.
    """
    print("="*80)
    print("INTERACTIVE GENERATION DEMO")
    print("="*80)
    print("\nThis demo shows how the system:")
    print("  1. Monitors entropy during generation")
    print("  2. Classifies uncertainty type when H > threshold")
    print("  3. Stops on CODE uncertainty to ask for clarification")
    print("  4. Continues on LANGUAGE uncertainty (just word choice)")
    print("\nTry these example prompts:")
    print("  - 'import' (code uncertainty - should stop)")
    print("  - 'The algorithm is' (language uncertainty - should continue)")
    print("  - 'SELECT * FROM' (code uncertainty - should stop)")
    print("="*80)
    
    while True:
        prompt = input("\nEnter prompt (or 'quit' to exit): ").strip()
        
        if prompt.lower() == 'quit':
            print("Exiting demo.")
            break
        
        if not prompt:
            print("Please enter a non-empty prompt.")
            continue
        
        result = generate_with_entropy_monitoring(
            prompt,
            entropy_threshold=3.0,
            max_tokens=30,
            verbose=True
        )
        
        if result['stop_reason'] == 'code_uncertainty':
            print("\n" + "="*80)
            print("❗ SYSTEM NEEDS CLARIFICATION")
            print("="*80)
            print(f"The model is uncertain about which code element to use.")
            print(f"Please provide more context or specify which {result['stop_info']['next_token_would_be']} to use.")
            print("="*80)

# Uncomment to run interactive demo
# interactive_generation_demo()

In [ ]:
# Cell 14: Final summary

print("\n" + "="*80)
print("WEEK 3: ENTROPY-DRIVEN INTERACTIVE GENERATION - COMPLETE")
print("="*80)

print(f"\n🎯 GOAL ACHIEVED")
print(f"   Train probe on diverse data → Monitor entropy → Classify uncertainty → Act!")

print(f"\n📊 RESULTS")
print(f"   Code examples that stopped: {code_stopped}/{len(code_results)}")
print(f"   Language examples (should NOT stop): {len(lang_results) - lang_stopped}/{len(lang_results)} continued")
print(f"   Average entropy:")
print(f"     - Code uncertainty: {avg_entropy_code:.2f} bits")
print(f"     - Language uncertainty: {avg_entropy_lang:.2f} bits")
print(f"     - Confident: {avg_entropy_conf:.2f} bits")

print(f"\n🚀 HOW IT WORKS")
print(f"   1. Generate tokens sequentially")
print(f"   2. At each step: compute entropy H")
print(f"   3. If H > {3.0} bits: classify uncertainty type")
print(f"   4. If CODE uncertainty: STOP and ask user")
print(f"   5. If LANGUAGE uncertainty: CONTINUE (just word choice)")

print(f"\n✅ KEY INNOVATIONS")
print(f"   - NO hardcoded keywords (100% probe-based)")
print(f"   - Entropy-driven (only checks when uncertain)")
print(f"   - Interactive stopping (asks for code context)")
print(f"   - Trained on {model_data['n_train_examples']} diverse examples")

print(f"\n📁 FILES")
print(f"   - uncertainty_type_classifier.pkl (trained probe)")
print(f"   - interactive_generation_results.png (visualizations)")

print(f"\n🎉 WEEK 3 COMPLETE!")
print(f"\n" + "="*80)